# 00 Colab Setup

Purpose: mount Google Drive, verify the mirrored Drive source folder, recreate the expected runtime and artifact folders when needed, install pinned runtime dependencies from that source, and resolve the persistent runtime paths.

Final workflow:
1. Author and validate code locally in the Git repo.
2. Rewrite or initialize the Drive folders locally with `make drive-rewrite-colab` or `make drive-init`.
3. Push the execution subset into Google Drive from the local machine with `make drive-push-source`.
4. Mount Drive in Colab.
5. Run this notebook.
6. Build the dataset manifests in Colab with `scripts/build_dataset_manifests.py --profile full`.
7. Execute the stage notebook you need from the Drive-backed source tree.

Persistence model:
- `/content/drive/MyDrive/json-ft-source` is the mirrored execution copy of the repo.
- `/content/drive/MyDrive/json-ft-runs` stores runtime outputs and survives Colab restarts.
- Source refresh happens on the local machine through explicit Drive sync commands, not through Colab upload staging.


In [1]:
from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
from pathlib import Path

SOURCE_ROOT = Path('/content/drive/MyDrive/json-ft-source')
RUNTIME_ROOT = Path('/content/drive/MyDrive/json-ft-runs')

required_paths = [
    SOURCE_ROOT / 'src',
    SOURCE_ROOT / 'scripts',
    SOURCE_ROOT / 'configs',
    SOURCE_ROOT / 'data',
    SOURCE_ROOT / 'notebooks',
    SOURCE_ROOT / 'scripts' / 'build_dataset_manifests.py',
    SOURCE_ROOT / 'scripts' / 'train_sft.py',
    SOURCE_ROOT / 'scripts' / 'build_preference_pairs.py',
    SOURCE_ROOT / 'scripts' / 'train_dpo.py',
    SOURCE_ROOT / 'scripts' / 'eval_model.py',
    SOURCE_ROOT / 'scripts' / 'compare_stages.py',
    SOURCE_ROOT / 'configs' / 'data_sources.yaml',
    SOURCE_ROOT / 'configs' / 'data_build.yaml',
    SOURCE_ROOT / 'configs' / 'sft.yaml',
    SOURCE_ROOT / 'configs' / 'dpo.yaml',
    SOURCE_ROOT / 'configs' / 'eval.yaml',
    SOURCE_ROOT / 'requirements-colab.txt',
]
source_dirs_to_create = [
    SOURCE_ROOT / 'artifacts' / 'metrics',
    SOURCE_ROOT / 'artifacts' / 'plots',
    SOURCE_ROOT / 'artifacts' / 'reports',
    SOURCE_ROOT / 'artifacts' / 'checkpoints',
]
runtime_dirs_to_create = [
    RUNTIME_ROOT / 'persistent' / 'metrics',
    RUNTIME_ROOT / 'persistent' / 'plots',
    RUNTIME_ROOT / 'persistent' / 'reports',
    RUNTIME_ROOT / 'persistent' / 'logs',
    RUNTIME_ROOT / 'persistent' / 'checkpoints',
    RUNTIME_ROOT / 'persistent' / 'exports',
    RUNTIME_ROOT / 'persistent' / 'preferences',
    RUNTIME_ROOT / 'persistent' / 'tokenized',
    RUNTIME_ROOT / 'scratch',
    RUNTIME_ROOT / 'raw-data',
]
missing = [path for path in required_paths if not path.exists()]

print(f'Source root: {SOURCE_ROOT}')
print(f'Runtime root: {RUNTIME_ROOT}')
if missing:
    missing_text = '\n'.join(str(path) for path in missing)
    raise FileNotFoundError(
        'Drive source is incomplete. Run make drive-push-source locally before continuing:\n' + missing_text
    )
for path in [*source_dirs_to_create, *runtime_dirs_to_create]:
    path.mkdir(parents=True, exist_ok=True)
print(f'Created or verified {len(source_dirs_to_create)} mirrored artifact directories and {len(runtime_dirs_to_create)} runtime directories.')
print('Drive source preflight passed.')


Source root: /content/drive/MyDrive/json-ft-source
Runtime root: /content/drive/MyDrive/json-ft-runs
Created or verified 4 mirrored artifact directories and 10 runtime directories.
Drive source preflight passed.


In [3]:
!python -m pip install --upgrade pip
!python -m pip install -r /content/drive/MyDrive/json-ft-source/requirements-colab.txt


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 2.5 MB/s eta 0:00:0000:0100:010m
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 50.0 MB/s  0:00:016m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 20.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 57.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 6.8 MB/s  0:00:01m0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 101.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 28.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.3/432.3 MB 35.7 MB/s  0:00:08m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 68.7 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 131.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
import sys

sys.path.insert(0, str(SOURCE_ROOT / 'src'))

from json_ft.runtime import resolve_runtime_context, format_runtime_summary

context = resolve_runtime_context(
    repo_root=SOURCE_ROOT,
    stage='setup',
    run_name='colab-session',
    runtime_root=RUNTIME_ROOT,
)
print(format_runtime_summary(context))
print('\nColab Drive-source setup is ready.')


is_colab=True
repo_root=/content/drive/MyDrive/json-ft-source
runtime_root=/content/drive/MyDrive/json-ft-runs
persistent_root=/content/drive/MyDrive/json-ft-runs/persistent
scratch_root=/content/drive/MyDrive/json-ft-runs/scratch
plots_dir=/content/drive/MyDrive/json-ft-runs/persistent/plots
stage=setup
run_name=colab-session
run_dir=/content/drive/MyDrive/json-ft-runs/persistent/setup/colab-session

Colab Drive-source setup is ready.
